# Geospatial ML — Train on Colab & Save to Drive

This notebook trains / prepares all 3 model checkpoints needed for the API:
1. **ResNet-50 classifier** — land-cover classification (`best_model.pt`)
2. **Convolutional Autoencoder** — anomaly detection (`autoencoder_best.pt`)
3. **Mask R-CNN** — tree crown segmentation (`segmentation_best.pt`)

Checkpoints are saved to your Google Drive under `MyDrive/geospatial_checkpoints/`.

> Make sure **Runtime → Change runtime type → T4 GPU** is selected before running.

## Step 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_OUTPUT = '/content/drive/MyDrive/geospatial_checkpoints'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
print(f'Checkpoints will be saved to: {DRIVE_OUTPUT}')

## Step 2 — Clone the repo

In [ ]:
%cd /content
!git clone https://github.com/iamvisheshsrivastava/geospatial
%cd geospatial
!git log --oneline -3

## Step 3 — Install dependencies

In [ ]:
!pip install -q \
    torch torchvision \
    rasterio \
    wandb \
    boto3 \
    scikit-learn \
    numpy pandas matplotlib pillow \
    pydantic pydantic-settings \
    tqdm

import torch
print(f'PyTorch {torch.__version__}')
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## Step 4 — Download & extract EuroSAT dataset (~90 MB)

Uses `requests` with SSL verification disabled — more reliable than `wget` because
we can validate the HTTP response and check the file is actually a ZIP before extracting.
Every valid ZIP starts with magic bytes `PK\x03\x04` — if the server returns an HTML
error page instead, we catch it immediately with a clear error.

Extraction searches the entire unpacked tree for class directories, so it works
regardless of how many nesting levels the ZIP uses internally.

In [ ]:
import os, zipfile, shutil, requests
from pathlib import Path
import urllib3

CLASSES = [
    'AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial',
    'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake'
]
EUROSAT_DIR = Path('data/eurosat')
EUROSAT_DIR.mkdir(parents=True, exist_ok=True)
zip_path = EUROSAT_DIR / 'EuroSAT.zip'

URL = 'https://madm.dfki.de/files/sentinel/EuroSAT.zip'
print(f'Downloading EuroSAT from {URL} ...')
resp = requests.get(URL, stream=True, timeout=120)
resp.raise_for_status()

total = int(resp.headers.get('content-length', 0))
downloaded = 0
with open(zip_path, 'wb') as f:
    for chunk in resp.iter_content(chunk_size=65536):
        f.write(chunk)
        downloaded += len(chunk)
        if total:
            pct = downloaded / total * 100
            print(f'\r  {downloaded // 1_048_576} / {total // 1_048_576} MB  ({pct:.1f}%)', end='')
print(f'\nDownload complete. File size: {zip_path.stat().st_size / 1_048_576:.1f} MB')

with open(zip_path, 'rb') as f:
    magic = f.read(4)
if magic != b'PK\x03\x04':
    raise RuntimeError(
        f'Downloaded file is NOT a ZIP (first 4 bytes: {magic}). '
        'The server likely returned an HTML error page.'
    )
print('ZIP magic bytes OK.')

print('\nZIP contents (first 20 entries):')
with zipfile.ZipFile(zip_path) as zf:
    for name in zf.namelist()[:20]:
        print(f'  {name}')

tmp_dir = EUROSAT_DIR / '_tmp_extract'
if tmp_dir.exists():
    shutil.rmtree(tmp_dir)
tmp_dir.mkdir()
print('\nExtracting...')
with zipfile.ZipFile(zip_path) as zf:
    zf.extractall(tmp_dir)

found = {}
for dirpath, dirnames, _ in os.walk(tmp_dir):
    for d in list(dirnames):
        if d in CLASSES and d not in found:
            found[d] = Path(dirpath) / d

print(f'Found {len(found)}/10 class directories.')
for cls, src in found.items():
    dest = EUROSAT_DIR / cls
    if dest.exists():
        shutil.rmtree(dest)
    shutil.move(str(src), str(dest))

shutil.rmtree(tmp_dir, ignore_errors=True)
zip_path.unlink(missing_ok=True)

print('\nDataset verification:')
all_ok = True
for cls in CLASSES:
    d = EUROSAT_DIR / cls
    count = len(list(d.glob('*'))) if d.exists() else 0
    status = 'OK' if count > 0 else 'MISSING'
    if count == 0:
        all_ok = False
    print(f'  {status:7s}  {cls} ({count} images)')

if not all_ok:
    raise RuntimeError('Some class directories are missing — check ZIP contents above.')
print('\nAll 10 classes ready.')

## Step 5 — Train the ResNet-50 classifier

~10 minutes on T4 GPU. Fine-tunes a ResNet-50 (pretrained on ImageNet) on EuroSAT
to classify satellite patches into 10 land-cover types. Saves the best checkpoint
automatically based on validation F1 score.

In [ ]:
!python -m src.train \
    --data-root data/eurosat \
    --epochs 10 \
    --batch-size 64 \
    --learning-rate 3e-4 \
    --num-workers 2 \
    --checkpoint-dir checkpoints \
    --wandb-mode disabled

## Step 6 — Train the Anomaly Detector (IEEE IJCNN 2024 methodology)

~20 minutes on T4 GPU. Implements the full methodology from:
>  V. Srivastava, "Autoencoder Optimization for Anomaly Detection," **IEEE IJCNN 2024**.

Key decisions applied automatically:
- **BCE loss** (over MSE) — proven better for coloured satellite images
- **[0, 1] normalisation** (not ImageNet stats)
- **3 architectures trained in parallel**: CAE-2Conv, CAE-3Conv, CAE-VariedFilter
- **Best model selected by AUC-ROC** on a normal/anomaly held-out split
- **Horizontal + vertical flip augmentation only** (no rotation — orientation matters)
- **Early stopping** patience = 5
- **Anomaly threshold** = 95th percentile of normal reconstruction errors (saved in checkpoint)


In [ ]:
!python -m src.anomaly 
--data-root data/eurosat 
--normal-classes Forest 
--epochs 50 
--batch-size 256 
--image-size 64 
--patience 5 
--threshold-percentile 95 
--wandb-mode disabled

# Check the saved checkpoint
import torch
ckpt = torch.load("checkpoints/autoencoder_best.pt", map_location="cpu", weights_only=False)
print(f"Best arch : {ckpt["arch"]}")
print(f"AUC-ROC   : {ckpt["auc_roc"]:.4f}")
print(f"Threshold : {ckpt["threshold"]:.6f}  (95th pct of normal errors)")


## Step 6b — Save Mask R-CNN for tree crown segmentation

Downloads a Mask R-CNN pretrained on COCO (80 object classes) from torchvision
and saves it as `segmentation_best.pt`. This gives the `/segment` endpoint a real
working model — it detects and segments objects in aerial imagery out of the box.
No custom training data needed.

In [ ]:
import torch, torchvision
from pathlib import Path

Path('checkpoints').mkdir(exist_ok=True)
print('Downloading pretrained Mask R-CNN weights...')
model = torchvision.models.detection.maskrcnn_resnet50_fpn(weights='DEFAULT')
torch.save(model.state_dict(), 'checkpoints/segmentation_best.pt')
size_mb = Path('checkpoints/segmentation_best.pt').stat().st_size / 1_048_576
print(f'Saved checkpoints/segmentation_best.pt ({size_mb:.1f} MB)')

## Step 7 — Verify all checkpoints were created

In [ ]:
import os
all_good = True
for f in ['checkpoints/best_model.pt', 'checkpoints/autoencoder_best.pt', 'checkpoints/segmentation_best.pt']:
    if os.path.exists(f):
        size_mb = os.path.getsize(f) / 1_048_576
        print(f'  OK       {f}  ({size_mb:.1f} MB)')
    else:
        print(f'  MISSING  {f}')
        all_good = False

print('\nAll 3 checkpoints ready — proceed to Step 8.' if all_good else '\nSome checkpoints missing — check steps above.')

## Step 8 — Upload checkpoints to Kaggle Dataset → triggers Heroku deploy

This cell:
1. Uploads the 3 model checkpoints as a **Kaggle Dataset** (`geospatial-checkpoints`)
2. Pushes a commit to GitHub → triggers GitHub Actions
3. GitHub Actions downloads from the dataset → builds Docker → deploys to Heroku

**No Google Drive needed. Works every time without committing the notebook.**

**Before running:** add `GITHUB_TOKEN` to Kaggle Secrets (Add-on → Secrets).

In [ ]:
import subprocess, json, datetime, shutil, os
from pathlib import Path
from kaggle_secrets import UserSecretsClient

GITHUB_TOKEN = UserSecretsClient().get_secret('GITHUB_TOKEN')
KAGGLE_TOKEN = UserSecretsClient().get_secret('KAGGLE_API_TOKEN') if 'KAGGLE_API_TOKEN' in [s.label for s in UserSecretsClient().get_secrets()] else None
REPO   = 'iamvisheshsrivastava/geospatial'
BRANCH = 'main'
RUN_TS = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%d_%H%M%S')

# Set up Kaggle API credentials
os.environ['KAGGLE_API_TOKEN'] = KAGGLE_TOKEN or ''

# ── 1. Verify checkpoints exist ───────────────────────────────────────────
FILES = ['best_model.pt', 'autoencoder_best.pt', 'segmentation_best.pt']
print('Verifying checkpoints...')
for fname in FILES:
    p = Path('checkpoints') / fname
    if not p.exists():
        raise FileNotFoundError(f'{fname} not found — run training cells first.')
    print(f'  OK  {fname}  ({p.stat().st_size / 1_048_576:.1f} MB)')

# ── 2. Upload to Kaggle Dataset ───────────────────────────────────────────
dataset_dir = Path('/kaggle/working/dataset_upload')
dataset_dir.mkdir(exist_ok=True)

# Copy checkpoints into upload folder
for fname in FILES:
    shutil.copy2(Path('checkpoints') / fname, dataset_dir / fname)

# Write dataset metadata
metadata = {
    'title': 'Geospatial ML Checkpoints',
    'id': 'visheshsrivastava/geospatial-checkpoints',
    'licenses': [{'name': 'CC0-1.0'}]
}
with open(dataset_dir / 'dataset-metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print(f'\nUploading checkpoints to Kaggle Dataset...')

# Try to create new dataset; if it exists, add a new version instead
result = subprocess.run(
    ['kaggle', 'datasets', 'create', '-p', str(dataset_dir), '--dir-mode', 'zip'],
    capture_output=True, text=True
)
if 'already exists' in result.stdout + result.stderr or result.returncode != 0:
    print('Dataset exists — uploading new version...')
    result = subprocess.run(
        ['kaggle', 'datasets', 'version', '-p', str(dataset_dir),
         '-m', f'Training run {RUN_TS}', '--dir-mode', 'zip'],
        capture_output=True, text=True
    )
print(result.stdout or result.stderr)

# ── 3. Push trigger commit to GitHub ─────────────────────────────────────
print('Pushing trigger commit to GitHub...')

repo_dir = Path('/kaggle/working/repo_push')
if repo_dir.exists():
    shutil.rmtree(repo_dir)

subprocess.run([
    'git', 'clone', '--depth', '1',
    f'https://{GITHUB_TOKEN}@github.com/{REPO}.git', str(repo_dir)
], check=True)

trigger = {
    '_trained_at': RUN_TS,
    '_dataset': 'visheshsrivastava/geospatial-checkpoints',
    '_note': 'Auto-updated by Kaggle notebook — do not edit manually.'
}
with open(repo_dir / 'model_config.json', 'w') as f:
    json.dump(trigger, f, indent=2)

subprocess.run(['git', '-C', str(repo_dir), 'config', 'user.email', 'kaggle-training@auto.bot'], check=True)
subprocess.run(['git', '-C', str(repo_dir), 'config', 'user.name', 'Kaggle Training Bot'], check=True)
subprocess.run(['git', '-C', str(repo_dir), 'add', 'model_config.json'], check=True)
subprocess.run([
    'git', '-C', str(repo_dir), 'commit', '--allow-empty', '-m',
    f'chore(models): training run {RUN_TS} — deploy from Kaggle dataset'
], check=True)
subprocess.run(['git', '-C', str(repo_dir), 'push', 'origin', BRANCH], check=True)

print('\n' + '='*60)
print(f'  DONE — run {RUN_TS}')
print('='*60)
print('Checkpoints uploaded to: kaggle.com/visheshsrivastava/geospatial-checkpoints')
print('GitHub Actions will now:')
print('  1. Download checkpoints from Kaggle Dataset')
print('  2. Build Docker image')
print('  3. Deploy to Heroku (~12 min)')
print('Watch: https://github.com/iamvisheshsrivastava/geospatial/actions')
print('='*60)


## Step 9 — Quick sanity check (optional)

Loads the trained classifier and runs a prediction on a random EuroSAT image.
If this prints a class name and confidence score, the model is working correctly.

In [ ]:
import torch
from pathlib import Path
from src.models.resnet import build_resnet50_classifier
from src.data.preprocessing import preprocess_image

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ckpt = torch.load('checkpoints/best_model.pt', map_location=device, weights_only=False)
class_names = ckpt['class_names']
model = build_resnet50_classifier(num_classes=len(class_names), pretrained=False)
model.load_state_dict(ckpt['model_state_dict'])
model.to(device).eval()

sample = next(Path('data/eurosat').glob('*/*.jpg'))
tensor = preprocess_image(sample, 224).unsqueeze(0).to(device)

with torch.no_grad():
    probs = torch.softmax(model(tensor), dim=1).squeeze()

conf, idx = probs.max(0)
print(f'Image     : {sample}')
print(f'Predicted : {class_names[idx]} ({conf:.1%} confidence)')
print(f'Val F1    : {ckpt["metrics"]["macro_f1"]:.4f}')